# Lab 13 · seaborn, bản đồ số chỗ ở và phản biện cách chọn mốc

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 13**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng bài 13 vẽ choropleth **giá** và phân tích ba biểu đồ lỗi. Trong lab này,
bạn sẽ vẽ bản đồ **số chỗ ở** để bổ sung cho bản đồ giá, so sánh phân phối bằng seaborn
theo một lát cắt mới và lập hồ sơ lỗi cho một hình chọn mốc thời gian có lợi.

*Lab khoảng 50 phút; 30 phút cuối dành cho phần hỗ trợ bài tập lớn ở cuối notebook.*

**Mục tiêu bài lab**

1. Dùng seaborn (boxplot và hue) để so sánh phân phối giữa các nhóm trên thang log.
2. Vẽ choropleth **số chỗ ở** và đặt cạnh bản đồ giá để bổ sung thông tin về quy mô nguồn cung.
3. Xử lý quận có mặt trên bản đồ nhưng không có trong bảng dữ liệu (NaN sau phép ghép).
4. Lập hồ sơ lỗi và sửa một hình chọn mốc so sánh có lợi.

Phiên bản đề: `seaborn-maps-critique-v1` (25/09/2026). Lưu đúng tên **`lab-13.ipynb`**.

## Cách làm và nộp bài

- Với phần có hướng dẫn, bạn nên **tự gõ, không dùng AI** — các bài kiểm tra định kỳ 🚫 đóng ở giờ lý thuyết
  kiểm tra những kỹ năng này. Bài tự làm cuối lab ở chế độ ✅ mở: được dùng AI, kèm khai báo.
- Điền thân hàm TODO và phần trả lời Markdown. Hàm vẽ **tạo hình mới** và **trả về `ax`**; grader đọc cấu trúc
  của `ax` (thang trục, chú giải, dữ liệu trên đường, vùng tô, tiêu đề) và file đã lưu.
- Giữ tên hàm, tham số, cell ID và metadata; viết helper/import thêm trong chính cell bài làm. Grader chỉ chạy cell câu đó
  với dữ liệu riêng, không phụ thuộc biến `ds`, `rv`, `geo`, `nam` ở cell khác. `pd`, `np` được cung cấp sẵn.
  Phần bản đồ được chấm ở **bước ghép dữ liệu** (bảng thường có cột `neighbourhood`); bản thân hình bản đồ do giảng viên xem.
- Public checks chỉ là ví dụ tự kiểm tra trên dữ liệu giả lập. Một check lỗi không dừng các check sau.
  TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa. Không sửa input; không ghi cứng kết quả.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1 (boxplot seaborn) 20 · Bài 2 (ghép bản đồ số chỗ ở) 20 | 40 | Tự động |
| Bài 3 (đếm theo năm) 10 · Bài 4 (so các mốc) 15 · Bài 5 (bản sửa hình) 15 | 40 | Tự động |
| Hồ sơ lỗi, diễn giải boxplot và cặp bản đồ, hình bản đồ | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Phần 🧭 hỗ trợ bài tập lớn ở cuối lab làm theo nhóm, không tính vào điểm lab.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu và thư viện

Lab cần `seaborn` và, cho phần vẽ bản đồ, `geopandas` (Colab có sẵn cả hai). Thiếu thư viện thì chạy
`%pip install seaborn geopandas` trong một cell riêng rồi Restart; các public check không cần geopandas.

Mặc định `USE_SNAPSHOT = False`: notebook dùng **dữ liệu giả lập** (48 chỗ ở ở 5 quận; bản đồ 6 quận; số đánh giá mỗi năm 2014–2026).
Đặt `USE_SNAPSHOT = True` để đọc `listings.csv`, `reviews.csv` và `neighbourhoods.geojson` của snapshot Santiago **2026-06-29**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    print("Thiếu seaborn: chạy %pip install seaborn trong một cell riêng rồi Restart.")

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations"

# Dữ liệu GIẢ LẬP: 48 chỗ ở ở 5 quận (bản đồ có 6 quận) và số đánh giá mỗi năm 2014–2026.
_QUAN = ["Santiago"] * 18 + ["Providencia"] * 12 + ["Las Condes"] * 8 + ["Ñuñoa"] * 6 + ["Lo Barnechea"] * 4
DEMO_DS = pd.DataFrame({
    "id": range(201, 249),
    "neighbourhood": _QUAN,
    "room_type": ["Private room" if k % 4 == 1 else "Entire home/apt" for k in range(48)],
    "price": [None if k % 11 == 5 else float(18000 + (k * 7919) % 90000 + (15000 if k % 3 == 0 else 0))
              for k in range(48)],
    "calculated_host_listings_count": [1 + (k * 5) % 9 for k in range(48)],
})
DEMO_QUAN_BAN_DO = ["Santiago", "Providencia", "Las Condes", "Ñuñoa", "Lo Barnechea", "Pirque"]
_SO_THEO_NAM = {2014: 40, 2015: 70, 2016: 120, 2017: 210, 2018: 330, 2019: 420, 2020: 90,
                2021: 150, 2022: 380, 2023: 520, 2024: 610, 2025: 700, 2026: 260}
DEMO_RV = pd.DataFrame({"date": [pd.Timestamp(f"{nam}-01-01") + pd.Timedelta(days=(k * 97) % (181 if nam == 2026 else 365))
                                 for nam, so in _SO_THEO_NAM.items() for k in range(so)]})

# Đặt USE_SNAPSHOT = True để làm trên snapshot Santiago thật.
USE_SNAPSHOT = False

if USE_SNAPSHOT:
    ds = pd.read_csv(f"{BASE}/listings.csv")
    rv = pd.read_csv(f"{BASE}/reviews.csv", parse_dates=["date"])
    DATA_LABEL = "Snapshot Santiago 2026-06-29"
else:
    ds, rv = DEMO_DS.copy(), DEMO_RV.copy()
    DATA_LABEL = "GIẢ LẬP — 48 chỗ ở, bản đồ 6 quận"

ds["kieu_host"] = (ds["calculated_host_listings_count"] >= 5).map({True: "chuyên nghiệp", False: "cá nhân"})
co_gia = ds.dropna(subset=["price"])
Path("figures").mkdir(exist_ok=True)
print(DATA_LABEL, "·", len(ds), "chỗ ở ·", len(rv), "đánh giá")

## Bài 1 · Boxplot và hue: giá theo nhóm chủ nhà

Biểu đồ so sánh giá của chủ nhà chuyên nghiệp (có từ 5 chỗ ở) với chủ nhà cá nhân, đồng thời tách theo loại phòng.
Cell dữ liệu đã tạo cột `kieu_host` và bảng `co_gia` (bỏ dòng thiếu giá).

### Hợp đồng hàm · 20 điểm tự động

`plot_price_box(co_gia: pd.DataFrame) -> Axes`

Nhận bảng có cột `price` (dương, không thiếu), `room_type`, `kieu_host`. Tạo hình mới `fig, ax = plt.subplots(figsize=(8.5, 4))`,
vẽ `sns.boxplot(data=co_gia, x="price", y="room_type", hue="kieu_host", ax=ax)`, đặt **thang log** cho trục x
(`ax.set_xscale("log")`), tiêu đề nêu thông điệp (≥ 15 ký tự), nhãn trục x có đơn vị, và trả về `ax`.
Chú giải (do `hue` sinh) phải liệt kê đúng các giá trị của `kieu_host`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_price_box(co_gia: pd.DataFrame):
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q1():
    demo = DEMO_DS.dropna(subset=["price"]).assign(
        kieu_host=lambda d: (d["calculated_host_listings_count"] >= 5).map({True: "chuyên nghiệp", False: "cá nhân"}))
    ax = plot_price_box(demo)
    assert ax.get_xscale() == "log", "Giá lệch phải — boxplot phải xem trên thang log"
    assert len(ax.get_title()) >= 15
    assert ax.get_legend() is not None, "hue phải sinh chú giải"
    nhan = {t.get_text() for t in ax.get_legend().get_texts()}
    assert nhan == {"chuyên nghiệp", "cá nhân"}
    assert len(ax.patches) >= 4, "Cần một hộp cho mỗi tổ hợp loại phòng × kiểu chủ nhà"
    plt.close(ax.figure)

public_check("Q1 · ví dụ công khai", check_q1)

In [ ]:
preview("Bài 1 trên dữ liệu đang dùng", lambda: plot_price_box(co_gia).get_title())

**Đọc boxplot:** ở từng loại phòng, hộp của nhóm chuyên nghiệp nằm lệch về phía nào? Hai nhóm có tách thành hai phân khúc riêng không? Vì sao cần thang log? …

## Bài 2 · Cặp bản đồ giá và số chỗ ở — bước ghép dữ liệu

Bản đồ giá dễ làm các quận có diện tích lớn thu hút quá nhiều sự chú ý. Đặt thêm bản đồ **số chỗ ở** bên cạnh sẽ cho biết quy mô
nguồn cung của từng quận. Một quận có thể có ranh giới trên bản đồ nhưng **không có chỗ ở nào** trong mốc chụp: sau phép ghép nó là NaN,
và ở đây NaN có nghĩa là **0** (không phải "chưa biết").

### Hợp đồng hàm · 20 điểm tự động

`count_by_area(ds: pd.DataFrame, geo: pd.DataFrame) -> dict`

Nhận bảng chỗ ở (cột `neighbourhood`) và bảng ranh giới quận `geo` (GeoDataFrame hoặc DataFrame có cột `neighbourhood` không trùng).
Đếm số chỗ ở mỗi quận thành bảng hai cột `neighbourhood`, `n`; `geo.merge(..., on="neighbourhood", how="left")`. Trả dict có đúng các khóa:

- `so_quan_nan`: int, số quận trên bản đồ **không có** chỗ ở nào (NaN ngay sau phép ghép);
- `ban_do`: bảng sau phép ghép — đủ các cột của `geo` rồi cột `n`, giữ số dòng và thứ tự dòng của `geo`, cột `n` đã **điền 0** và là số nguyên.

Chỗ ở thuộc quận không có trên bản đồ không xuất hiện trong `ban_do`. Không sửa `geo`.
Đối chiếu snapshot: 32 quận trên bản đồ, **1** quận NaN; quận nhiều chỗ ở nhất có **7.182**.

In [ ]:
def count_by_area(ds: pd.DataFrame, geo: pd.DataFrame) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q2():
    geo_demo = pd.DataFrame({"neighbourhood": DEMO_QUAN_BAN_DO, "neighbourhood_group": None})
    result = count_by_area(DEMO_DS, geo_demo)
    assert set(result) == {"ban_do", "so_quan_nan"}
    ban_do = result["ban_do"]
    assert len(ban_do) == 6 and list(ban_do.columns) == ["neighbourhood", "neighbourhood_group", "n"]
    assert result["so_quan_nan"] == 1, "Pirque có ranh giới nhưng không có chỗ ở"
    assert ban_do["n"].isna().sum() == 0 and ban_do["n"].max() == 18
    assert ban_do.loc[ban_do["neighbourhood"] == "Pirque", "n"].item() == 0
    assert "n" not in geo_demo.columns

public_check("Q2 · ví dụ công khai", check_q2)

In [ ]:
def ve_ban_do_so_cho_o():
    import geopandas as gpd
    if USE_SNAPSHOT:
        geo = gpd.read_file(f"{BASE}/neighbourhoods.geojson")
    else:
        from shapely.geometry import box
        o = [box(x, y, x + 1, y + 1) for x, y in [(0, 0), (1, 0), (2, 0), (0, 1), (1, 1), (2, 1)]]
        geo = gpd.GeoDataFrame({"neighbourhood": DEMO_QUAN_BAN_DO, "neighbourhood_group": None}, geometry=o)
    kq = count_by_area(ds, geo)
    ban_do = kq["ban_do"]
    fig, ax = plt.subplots(figsize=(7, 7))
    ban_do.plot(column="n", cmap="Blues", legend=True, edgecolor="#999", ax=ax)
    dau = ban_do.loc[ban_do["n"].idxmax(), "neighbourhood"]
    ax.set_title(f"Nguồn cung tập trung ở {dau}; {kq['so_quan_nan']} quận không có chỗ ở (tô 0)")
    ax.set_axis_off()
    fig.text(0.02, 0.02, f"n = {int(ban_do['n'].sum()):,} chỗ ở · quận không có chỗ ở được tô giá trị 0", fontsize=8)
    fig.savefig("figures/ban_do_so_cho_o.png", dpi=150, bbox_inches="tight")
    return f"{len(ban_do)} quận trên bản đồ, {kq['so_quan_nan']} quận trắng dữ liệu; nhiều nhất {dau}: {ban_do['n'].max():,}"

preview("Bài 2 — bản đồ số chỗ ở", ve_ban_do_so_cho_o)

**Cặp bản đồ:** đặt bản đồ số chỗ ở cạnh bản đồ giá của bài giảng cho thấy điều gì mà bản đồ giá không thể hiện? Vì sao ở đây NaN được đổi thành 0? …

## Bước 3 · Hồ sơ lỗi cho hình chọn mốc có lợi

Ô dưới dùng số liệu đúng nhưng chọn mốc bắt đầu làm kết luận bị phóng đại.

In [ ]:
def hinh_chon_moc_co_loi():
    nam = rv.set_index("date").resample("YE").size()
    tu_2020 = nam.loc["2020":"2025"]
    he_so = nam.loc["2025-12-31"] / nam.loc["2020-12-31"]
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot(tu_2020.index.year, tu_2020.values, marker="o", color="#E62727", lw=2)
    ax.set_title(f"BÙNG NỔ x{he_so:.0f}: đánh giá tăng {he_so:.0f} lần kể từ 2020!")
    plt.show()
    return tu_2020

preview("Hình gốc (chọn mốc có lợi)", hinh_chon_moc_co_loi)

## Bài 3 · Số đánh giá theo năm trong một khoảng

### Hợp đồng hàm · 10 điểm tự động

`yearly_counts(rv: pd.DataFrame, bat_dau: str, ket_thuc: str) -> pd.Series`

Nhận bảng có cột `date` (`datetime64`) và hai năm dạng chuỗi (ví dụ `"2016"`, `"2025"`). Trả
`rv.set_index("date").resample("YE").size()` cắt lát `.loc[bat_dau:ket_thuc]` — index là ngày cuối năm, năm không có đánh giá là 0.
Không sửa input.

In [ ]:
def yearly_counts(rv: pd.DataFrame, bat_dau: str, ket_thuc: str) -> pd.Series:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q3():
    nam = yearly_counts(DEMO_RV, "2016", "2025")
    assert isinstance(nam, pd.Series) and isinstance(nam.index, pd.DatetimeIndex)
    assert nam.index.year.tolist() == list(range(2016, 2026))
    assert nam.loc["2020-12-31"] == 90 and nam.loc["2025-12-31"] == 700
    assert yearly_counts(DEMO_RV, "2026", "2026").tolist() == [260]

public_check("Q3 · ví dụ công khai", check_q3)

## Bài 4 · Cùng một năm cuối, đổi mốc so sánh

### Hợp đồng hàm · 15 điểm tự động

`compare_baselines(nam: pd.Series, nam_cuoi: int, cac_moc: list) -> dict`

Nhận Series số đếm theo năm (index ngày cuối năm), năm cuối và danh sách năm làm mốc (đều có trong `nam`). Trả dict
`{moc: số đếm năm nam_cuoi / số đếm năm moc}` cho từng mốc theo thứ tự của `cac_moc`, giá trị float không làm tròn.
Chọn phần tử theo năm, ví dụ `nam[nam.index.year == 2020].iloc[0]`. Không sửa input.

In [ ]:
def compare_baselines(nam: pd.Series, nam_cuoi: int, cac_moc: list) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    nam = DEMO_RV.set_index("date").resample("YE").size()
    result = compare_baselines(nam, 2025, [2020, 2019, 2016])
    assert list(result) == [2020, 2019, 2016]
    assert round(result[2020], 2) == 7.78, "700 / 90"
    assert round(result[2019], 2) == 1.67 and round(result[2016], 2) == 5.83

public_check("Q4 · ví dụ công khai", check_q4)

## Bài 5 · Bản sửa trung thực

### Hợp đồng hàm · 15 điểm tự động

`plot_fixed_trend(nam: pd.Series, path, covid: tuple = (2020, 2021)) -> Axes`

Nhận Series số đếm theo năm (chỉ gồm các năm **trọn**, ví dụ 2016–2025), đường dẫn PNG và khoảng năm COVID. Tạo hình mới và:

- vẽ **một đường** với đúng mọi giá trị của `nam` (trục x là năm: `nam.index.year`);
- **đánh dấu giai đoạn COVID** bằng vùng tô `ax.axvspan(covid[0] - 0.5, covid[1] + 0.5, ...)` hoặc chú thích chữ `ax.annotate/ax.text`;
- tiêu đề phù hợp với dữ liệu (≥ 15 ký tự, không phóng đại), nhãn trục y;
- `fig.savefig(path, dpi=150, bbox_inches="tight")` và trả về `ax`.

In [ ]:
import matplotlib.pyplot as plt

def plot_fixed_trend(nam: pd.Series, path, covid: tuple = (2020, 2021)):
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    import tempfile
    du_lieu_sua = DEMO_RV.set_index("date").resample("YE").size().loc["2016":"2025"]
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "trend.png"
        ax = plot_fixed_trend(du_lieu_sua, path)
        assert path.exists() and path.stat().st_size > 0, "Cần lưu file PNG"
    assert any(list(l.get_ydata()) == du_lieu_sua.tolist() for l in ax.lines), \
        "Bản sửa phải vẽ đủ các năm 2016–2025 — dài hơn hình gốc 2020–2025"
    assert len(ax.patches) + len(ax.texts) >= 1, "Cần đánh dấu giai đoạn COVID (axvspan hoặc chú thích)"
    assert len(ax.get_title()) >= 15 and ax.get_ylabel() != ""
    plt.close(ax.figure)

public_check("Q5 · ví dụ công khai", check_q5)

In [ ]:
def ban_sua():
    nam = yearly_counts(rv, "2016", "2025")
    ti_le = compare_baselines(nam, 2025, [2020, 2019])
    ax = plot_fixed_trend(nam, "figures/danh_gia_theo_nam.png")
    return (f"2025 so với 2020: x{ti_le[2020]:.1f} · so với 2019: x{ti_le[2019]:.2f}\n{ax.get_title()}")

preview("Bài 3–5 — bản sửa", ban_sua)

**Hồ sơ lỗi của hình gốc:**

- Lỗi 1 (mốc so sánh): …
- Lỗi 2 (lời so với số): …
- Bản sửa khác hình gốc ở đâu: …

## Bài tự làm ✅ mở — không cộng vào 80 điểm tự động

**Tự làm 1 · Biểu đồ tương tác.** Dùng `plotly.express.scatter` vẽ kinh độ, vĩ độ của 5.000 chỗ ở
với `hover_name="name"`, `hover_data=["neighbourhood", "price"]`. Kiểm tra năm điểm
có giá cao bất thường và viết hai dòng về thông tin thao tác rê chuột cung cấp, cùng lý do báo cáo
PDF vẫn cần hình tĩnh.

**Tự làm 2 · Cặp bản đồ cho thành phố của nhóm.** Vẽ cặp choropleth gồm giá trung vị và số chỗ ở
cho thành phố bài tập lớn của nhóm. Mỗi mốc chụp Inside Airbnb đều có `neighbourhoods.geojson`.
Hãy xử lý quận không có chỗ ở và ghi cỡ mẫu `n` trong chú thích.

In [ ]:
# Viết bài tự làm của bạn ở đây (không được chấm tự động).

**Bài tự làm của tôi:** …

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu số quận, tỷ số hoặc thang trục đã đối chiếu.

---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm theo nhóm, không tính điểm lab)

Trọng tâm: **kiểm tra chất lượng hình và tiến độ hợp phần LLM.**

1. ☐ Dùng **năm câu hỏi phản biện** ở slide 13 để kiểm tra từng hình của nhóm:
   trục, đơn vị và thang đo, cỡ mẫu n, dạng biểu đồ, lời diễn giải. Ghi lại hình chưa đạt.
2. ☐ Nếu có hình bản đồ: đã kèm bản đồ số chỗ ở hoặc chú thích cỡ mẫu n chưa?
3. ☐ Hợp phần LLM: đã gán tay ít nhất 100 nhãn, mỗi đánh giá do hai người gán;
   đã có kết quả độ chính xác đầu tiên của LLM và baseline.
4. ☐ So sánh giữa các mốc chụp bằng cùng thang đo; quy về chỉ số
   khi hai thành phố dùng đơn vị tiền tệ khác nhau, như ví dụ Hình C ở slide bài 13.
5. ☐ Kế hoạch hai tuần cuối nêu rõ người phụ trách từng mục và hạn hoàn thành bản nháp
   (khuyến nghị: trước bài 14 ba ngày).

> Nhóm hoàn thành sớm: đổi hình cho nhóm bên cạnh và phản biện chéo bằng năm câu hỏi.

## Tóm tắt bài lab

| Bạn đã làm | Dùng cho |
|---|---|
| Boxplot và hue trên thang log | so sánh phân phối nhiều chiều trong báo cáo |
| Cặp bản đồ giá và số chỗ ở; đổi NaN thành 0 có chủ đích | bổ sung quy mô nguồn cung cho bản đồ giá |
| Hồ sơ lỗi và bản sửa hình chọn mốc có lợi | phản biện hình do AI sinh (bài 13–14) |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

Bài giảng tiếp theo: **kể chuyện bằng dữ liệu và thẩm định báo cáo do AI tạo**.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-13.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_13.ipynb`).
2. Hoàn thành hàm TODO, hồ sơ lỗi, các diễn giải và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`. Không commit `figures/`.
4. Tạo tag `submit/lab-13/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-13` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()